# ElenchusEnv — GRPO Training with Qwen2.5-3B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eliot-gtn/syllogym/blob/main/notebooks/train_elenchus_grpo.ipynb)

Train **Qwen2.5-3B-Instruct** on **ElenchusEnv** — a multi-turn agentic deductive reasoning environment — using GRPO (Group Relative Policy Optimization).

The agent learns to use MCP tools (`check_rule`, `get_facts`, `derive`, `submit_answer`) to solve logical problems step by step.

**Datasets covered:** SyllogismGenerator (procedural), Knights & Knaves, ProofWriter d2-d5, FOLIO, FOL-NLI, LegalBench (hearsay + abercrombie)

**Recommended runtime:** A100 (40GB) or H100


## 1. Install dependencies

In [ ]:
# Run without %%capture the first time to catch any errors
import os

# 1. Downgrade torch to 2.6 for vLLM/Unsloth ABI compatibility
os.system('pip install -q "torch==2.6.0" torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124')

# 2. Install base deps BEFORE registering any local module
os.system('pip install -q openenv-core websockets "datasets>=2.19.0,<3.0.0" huggingface_hub matplotlib fastmcp')

# 3. Install Unsloth (picks up torch 2.6 installed above)
os.system('pip install -q --upgrade uv')
os.system('uv pip install -q "unsloth[huggingface]"')

print('Installation complete.')

In [ ]:
import os, sys, importlib, importlib.util

# Clone the repo (skip if already present)
REPO_NAME   = 'syllogym'
GH_USERNAME = 'eliot-gtn'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone https://github.com/{GH_USERNAME}/{REPO_NAME}.git')

# Register elenchus_env as an importable module
ENV_PATH = f'{REPO_NAME}/envs/elenchus_env'
spec = importlib.util.spec_from_file_location(
    'elenchus_env',
    f'{ENV_PATH}/__init__.py',
    submodule_search_locations=[ENV_PATH]
)
mod = importlib.util.module_from_spec(spec)
sys.modules['elenchus_env'] = mod
spec.loader.exec_module(mod)

print('elenchus_env registered.')

## 2. Configuration

In [ ]:
import warnings, logging
import torch

warnings.filterwarnings('ignore', category=FutureWarning, module='transformers')
logging.getLogger('transformers').setLevel(logging.ERROR)

# ── Environment ────────────────────────────────────────────────────────────────
ELENCHUS_URL  = 'https://farffadet-elenchus-env.hf.space'  # HF Space URL
HF_USERNAME   = 'farffadet'

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME    = 'unsloth/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN   = 4096   # longer context for multi-turn tool call history

# ── Training ───────────────────────────────────────────────────────────────────
MAX_STEPS     = 200
BATCH_SIZE    = 4
NUM_GEN       = 8      # completions per prompt for GRPO
LEARNING_RATE = 5e-6

# ── Eval ───────────────────────────────────────────────────────────────────────
EVAL_EPISODES = 30     # episodes per task for baseline/post-training eval
MAX_TOOL_STEPS = 6     # max tool calls per eval episode

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 3. Load model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=64,
    gpu_memory_utilization=0.9,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=64,
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded.')

## 4. Agentic helpers

In [ ]:
import re, json, torch
from vllm import SamplingParams
from openenv.core.env_client import EnvClient
from openenv.core.env_server.mcp_types import CallToolAction, ListToolsAction

# ── System prompt ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert logical reasoner. You solve deductive reasoning problems step by step using tools.

Available tools:
- check_rule()                        → get the rule/principle for this problem
- get_facts()                         → get the facts/premises
- derive(statement: str)              → record an intermediate deduction
- submit_answer(answer: str)          → submit your final answer (ends the episode)

Strategy:
1. Call check_rule() to understand the governing principle
2. Call get_facts() to read the premises
3. Call derive() one or more times to build your reasoning chain
4. Call submit_answer() with your final answer

Always respond with a single tool call in this exact JSON format:
{"tool": "tool_name", "arguments": {"arg_name": "arg_value"}}
For tools with no arguments use: {"tool": "check_rule", "arguments": {}}
"""

def extract_tool_call(text: str) -> dict | None:
    """Extract the first JSON tool call from model output."""
    match = re.search(r'\{[^{}]*"tool"[^{}]*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def run_episode(
    model, tokenizer,
    env_url: str = ELENCHUS_URL,
    task_mode: str = 'mixed',
    task_name: str | None = None,
    max_steps: int = MAX_TOOL_STEPS,
) -> float:
    """
    Run one full episode: connect → reset → tool calls → submit_answer.
    Returns the reward (1.0 correct, 0.0 wrong/timeout).
    """
    env = EnvClient(env_url)
    env.connect()
    try:
        kwargs = {}
        if task_name:
            kwargs = {'task_mode': 'single', 'task_name': task_name}
        reset_result = env.reset(**kwargs)
        obs = reset_result.observation
        problem = obs.get('problem', str(obs)) if isinstance(obs, dict) else str(obs)

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Problem:\n{problem}\n\nStart by calling check_rule().'},
        ]

        FastLanguageModel.for_inference(model)
        reward = 0.0

        for step in range(max_steps):
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors='pt', return_dict=True,
            ).to('cuda')
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=256, use_cache=True,
                    do_sample=True, temperature=0.7, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
            ).strip()

            call = extract_tool_call(response)
            if call is None:
                call = {'tool': 'submit_answer', 'arguments': {'answer': ''}}

            tool_name = call.get('tool', 'submit_answer')
            arguments  = call.get('arguments', {})

            result = env.step(CallToolAction(tool_name=tool_name, arguments=arguments))
            tool_output = ''
            if result.observation and result.observation.result:
                content = result.observation.result.content
                tool_output = content[0].text if content else ''

            messages.append({'role': 'assistant', 'content': response})
            messages.append({'role': 'user', 'content': f'[Tool result: {tool_output}]'})

            if tool_name == 'submit_answer' or result.done:
                reward = result.reward or 0.0
                break

        return reward
    finally:
        env.disconnect()

print('Helpers defined.')


## 5. Baseline evaluation (before training)

In [ ]:
import numpy as np

EVAL_TASKS = [
    'syllogism_d1', 'syllogism_d3', 'syllogism_d5',
    'knights_knaves',
    'proofwriter_d2', 'proofwriter_d4',
    'folio',
    'hearsay', 'abercrombie',
]

def eval_task(task_name: str, n_episodes: int = EVAL_EPISODES) -> float:
    rewards = [
        run_episode(model, tokenizer, task_name=task_name)
        for _ in range(n_episodes)
    ]
    return float(np.mean(rewards))

print('=== Baseline Evaluation (before training) ===')
baseline_scores = {}
for task in EVAL_TASKS:
    acc = eval_task(task, n_episodes=10)
    baseline_scores[task] = acc
    print(f'  {task}: {acc:.1%}')

print(f'\nMean baseline accuracy: {np.mean(list(baseline_scores.values())):.1%}')


## 6. Collect training prompts from ElenchusEnv

In [ ]:
DATASET_SIZE = 500

print(f'Collecting {DATASET_SIZE} problems from ElenchusEnv...')
train_prompts = []

env = EnvClient(ELENCHUS_URL)
env.connect()
try:
    for i in range(DATASET_SIZE):
        result = env.reset()
        obs = result.observation
        problem = obs.get('problem', '') if isinstance(obs, dict) else str(obs)
        train_prompts.append(
            tokenizer.apply_chat_template(
                [
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f'Problem:\n{problem}\n\nStart by calling check_rule().'},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        )
        if (i + 1) % 100 == 0:
            print(f'  {i + 1}/{DATASET_SIZE}')
finally:
    env.disconnect()

print(f'Collected {len(train_prompts)} prompts.')
print('\nExample prompt (first 400 chars):')
print(train_prompts[0][:400])

## 7. Reward functions

In [ ]:
import re

def reward_correct(completions, env_rewards=None, **kwargs):
    """
    Primary reward: 1.0 if the environment returned reward=1.0, else 0.0.
    env_rewards is populated by running the completion through the environment.
    """
    if env_rewards is None:
        return [0.0] * len(completions)
    return [float(r) for r in env_rewards]

def reward_tool_format(completions, **kwargs):
    """
    Small shaping reward (0.1) for producing valid JSON tool calls.
    Encourages the model to learn the tool call format early in training.
    Removed once training is stable.
    """
    rewards = []
    for completion in completions:
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)
        has_json = bool(re.search(r'\{[^{}]*"tool"[^{}]*\}', text))
        rewards.append(0.1 if has_json else 0.0)
    return rewards

print('Reward functions defined.')

## 8. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

FastLanguageModel.for_training(model)

train_dataset = Dataset.from_dict({'prompt': train_prompts})

training_args = GRPOConfig(
    use_vllm=True,
    learning_rate=LEARNING_RATE,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=5,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_generations=NUM_GEN,
    max_prompt_length=2048,
    max_completion_length=512,
    max_steps=MAX_STEPS,
    save_steps=50,
    max_grad_norm=0.1,
    report_to='none',
    output_dir='elenchus_grpo_checkpoints',
    # GRPO stability
    beta=0.001,           # KL penalty coefficient
    epsilon=0.2,          # PPO clip
    scale_rewards=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_correct, reward_tool_format],
    args=training_args,
    train_dataset=train_dataset,
)

print('Starting GRPO training...')
trainer.train()

## 9. Post-training evaluation & comparison

In [ ]:
import matplotlib.pyplot as plt

model.save_lora('elenchus_grpo_lora')

print('=== Post-Training Evaluation ===')
trained_scores = {}
for task in EVAL_TASKS:
    acc = eval_task(task, n_episodes=10)
    trained_scores[task] = acc
    delta = acc - baseline_scores.get(task, 0)
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'  {task}: {acc:.1%} ({arrow} {delta:+.1%})')

print(f'\nMean baseline:  {np.mean(list(baseline_scores.values())):.1%}')
print(f'Mean trained:   {np.mean(list(trained_scores.values())):.1%}')
print(f'Mean delta:     {np.mean([trained_scores[t] - baseline_scores.get(t, 0) for t in EVAL_TASKS]):+.1%}')

# Plot
tasks = list(EVAL_TASKS)
x = range(len(tasks))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i - width/2 for i in x], [baseline_scores.get(t, 0) for t in tasks],
       width, label='Baseline', color='#6c8ebf', alpha=0.85)
ax.bar([i + width/2 for i in x], [trained_scores.get(t, 0) for t in tasks],
       width, label='After GRPO', color='#82b366', alpha=0.85)
ax.set_xlabel('Task')
ax.set_ylabel('Accuracy')
ax.set_title('ElenchusEnv: Baseline vs Post-GRPO (Qwen2.5-3B)')
ax.set_xticks(list(x))
ax.set_xticklabels(tasks, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('elenchus_results.png', dpi=150)
plt.show()

## 10. Push to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

REPO_ID = f'{HF_USERNAME}/elenchus-qwen25-3b-grpo'

model.save_pretrained('elenchus_qwen25_3b_grpo')
tokenizer.save_pretrained('elenchus_qwen25_3b_grpo')
model.push_to_hub(REPO_ID, token=True)
tokenizer.push_to_hub(REPO_ID, token=True)

api = HfApi()
api.upload_file(
    path_or_fileobj='elenchus_results.png',
    path_in_repo='elenchus_results.png',
    repo_id=REPO_ID,
    repo_type='model',
)

print(f'Model pushed to: https://huggingface.co/{REPO_ID}')

## References

- **ElenchusEnv**: Multi-turn agentic deductive reasoning environment. [HF Space](https://huggingface.co/spaces/farffadet/elenchus-env)
- **GRPO**: Shao et al. (2024). *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*.
- **Unsloth**: Unsloth AI. [unsloth.ai](https://unsloth.ai)
- **OpenEnv**: Meta PyTorch. [GitHub](https://github.com/meta-pytorch/OpenEnv)
- **ProofWriter**: Tafjord et al. (2021). *ProofWriter: Generating Implications, Proofs, and Abductive Statements over Natural Language*.
- **FOLIO**: Han et al. (2022). *FOLIO: Natural Language Reasoning with First-Order Logic*. EMNLP.
- **LegalBench**: Guha et al. (2023). *LegalBench: A Collaboratively Built Benchmark for Measuring Legal Reasoning in Large Language Models*. NeurIPS.
